<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.1

### Miguel A. Bessa | <a href = "mailto: miguel_bessa@brown.edu">miguel_bessa@brown.edu</a>  | Associate Professor

### Elvis Aguero | <a href = "mailto: elvis_vera@brown.edu">elvis_vera@brown.edu</a>  | PhD candidate


**What:** A lecture of the "3dasm" course

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for entire course:** Murphy, Kevin P. *Probabilistic machine learning: an
introduction*. MIT press, 2022. Available online [here](https://probml.github.io/pml-book/book1.html)

**How:** We try to follow Murphy's book closely, but the sequence of Chapters and Sections is
different. The intention is to use notebooks as an introduction to the topic and Murphy's book
as a resource.
* If working offline: Go through this notebook and read the book.
* If attending class in person: listen to me (!) but also go through the notebook in your laptop at the same time. Read the book.
* If attending lectures remotely: listen to me (!) via Zoom and (ideally) use two screens where you have the notebook open in 1 screen and you see the lectures on the other. Read the book.

This is the first of two lectures on **adda**. Today: how you build such a framework, and what
we learned building it. Lecture 19.2: what happened when we pointed it at a real problem.

## **OPTION 1**. Run this notebook **locally in your computer**:
1. Confirm that you have the '3dasm' mamba (or conda) environment (see Lecture 1).
2. Go to the 3dasm_course folder in your computer and pull the last updates of the [repository](https://github.com/bessagroup/3dasm_course):
```
git pull
```
    - Note: if you can't pull the repo due to conflicts (and you can't handle these conflicts), use this command (with **caution**!) and your repo becomes the same as the one online:
```
git reset --hard origin/main
```
3. Open command window and load jupyter notebook (it will open in your internet browser):
```
jupyter notebook
```
5. Open notebook of this Lecture and choose the '3dasm' kernel.

## **OPTION 2**. Use **Google's Colab** (no installation required, but times out if idle):

1. go to https://colab.research.google.com
2. login
3. File > Open notebook
4. click on Github (no need to login or authorize anything)
5. paste the git link: https://github.com/bessagroup/3dasm_course
6. click search and then click on the notebook for this Lecture.

In [1]:
# Basic plotting tools needed in Python.

import matplotlib.pyplot as plt # import plotting tools to create figures
import numpy as np # import numpy to handle a lot of things!

%config InlineBackend.figure_format = "retina" # render higher resolution images in the notebook
plt.rcParams["figure.figsize"] = (8,4) # rescale figure size appropriately for slides

# To limit the number of rows to show in a dataframe, for presentation purposes:
import pandas as pd

pd.set_option('display.max_rows', 10)

In [2]:
# In Google Colab you need to install f3dasm first (locally it is already in the '3dasm'
# environment). Uncomment the line below if you are running in Colab:

# %pip install f3dasm

from f3dasm import ExperimentData   # the same object you used in Lectures 17, 18 and 19

## Outline for today

* Where we are: the data-driven process, and who makes the decisions
* What makes *design* different, and why that forces every choice we made
* Six concepts
    - the graph, delegation, the metered oracle and the ledger
    - falsification instead of a reward
    - the reproduction gate, and guards chosen by reversibility
* **In-class exercise**: be the reproduction gate
* Four lessons learned

**Reading material**: this notebook + the
[a3dasm documentation](https://elvis-aguero.github.io/a3dasm/).

*(A note on names: the framework is called **adda**: agentic data-driven design and analysis.
The Python package that implements it is `a3dasm`.)*

## Recap: the data-driven process

<img src=../figures/f3dasm_overview.svg width=90%>

In Lectures 17-19 you built each of these blocks yourself, with `f3dasm`: a `Domain`, an
`ExperimentData`, a `DataGenerator`, a sampler, an optimizer.

The framework standardizes the blocks. *You* supplied every decision: which parametrization,
which sampler, which surrogate, which optimizer, when to stop, and whether the answer you got
is real.

## The question behind today's lecture

Which of those are *decisions*, and which are *labor*?

<br>

Fitting a GP to 200 points is labor. Choosing to fit a GP at all is a decision. Writing an
Abaqus input deck is labor. Deciding that a new *shape* is worth a week of compute is a decision.

**adda** is a bet that an LLM-driven system can carry the labor and a useful share of the
decisions, provided we build the scaffolding that keeps it honest.

Resist presenting this as "we automated science". The honest claim is narrower: much of what a
graduate student does in a data-driven study is mechanical, and a good deal of what looks like
judgement is pattern-matching against literature and prior runs. Those parts transfer. What does
not transfer easily is knowing when you are being fooled, which is where most of the
engineering effort went.

## Designing a supercompressible metamaterial

<img src=../figures/supercompressible_schematic.png width=25% align='right'>

You want a slender lattice that compresses to a small fraction of its height and springs back
undamaged. You choose how many longerons it has, the cross-section of each one, and the support
rings that join them. Every candidate is scored by a finite element simulation, because the load
it buckles at is not something you can work out by hand.

Our problem has three features that shaped everything below.

**1. There is no reference solution.** You can measure any design: `sigma_crit`, `mcs`, `mls`.
What you cannot measure is how far you are from the best design, because nobody knows what it is.
So a claim about the design space cannot be checked against the right answer.

**2. Evaluating one design is expensive, and sometimes fails.** One Abaqus post-buckling solve is
minutes to hours, needs a license from a shared pool, and may not converge. Some evaluations
come back *partial*.

**3. The design space has no fixed boundary.** Thirteen variables is a decision someone made.
You can add a variable, change the topology, or invent a different parametrization, and each one
is a different space. Choosing which space to search is the research, and you cannot sample your
way to it.

Note: you might also be interested in other agentic frameworks that have made real progress on
scientific computing tasks. A good recent example is **GRAFT-ATHENA** (Toscano, Chai &
Karniadakis, 2026), which searches over *numerical methods* and reuses what worked on earlier
problems.

A method can be scored against a reference solution. For the mast there is none, which is why
our machinery goes into deciding what counts as evidence.

## Each feature forces a mechanism

**No reference solution** &rarr; a claim has to survive an attempt to refute it, because it cannot
be checked against the right answer.

**Expensive, flaky evaluations** &rarr; the simulator is reached through one metered entry point
that counts every call and keeps partial results.

**Evaluations are the scarce resource** &rarr; every stored result records where it came from:
which request produced it, when, and with what inputs.

**No fixed design space** &rarr; *we* write down what counts as an answer: the feasibility
limits, what may vary, what may not. The critic checks the work against that document, and you
will see the document itself next lecture.

Three facts about the problem, three mechanisms. Change the facts and you would build something
else.

This is the spine of the lecture. If a student remembers one slide, this is the one:
architecture as a consequence of what you can and cannot measure.

Note the framing: we are describing the constraints *we* face. Other frameworks solve different
problems and their choices follow from their own constraints, exactly as ours do from ours.

## Concept 1: From f3dasm blocks to a graph of agents

In Lectures 17-19 you wrote a fixed sequence: sample, fit, optimize, report. It cannot react, and
what you do next depends on what just happened.

Run the study by hand and you do four jobs: read what has already been tried, build the thing that
scores a design, write and run the search, and check whether the result holds up.

adda gives each job to its own agent, plus one hub that decides which job to do next.

* **literature reviewer**: finds and reads relevant papers
* **data generator**: turns a way of evaluating a design into a metered oracle
* **implementer**: writes and runs the code: sampling, surrogates, optimization
* **critic**: an adversarial reviewer that looks for holes before a result is accepted

Notice the implementer's job description. It is what *you* did in Lectures 18 and 19:
`create_sampler`, `create_optimizer`, `.arm(...)`, `.call(...)`.

<img src=../figures/adda_graph.png width=88%>

**Open loop**: there is no script. After every report the strategizer looks at the state and
chooses the next move. The loop ends when it declares the work done *and* that decision survives
review.

## Concept 2: Delegation is the unit of work *and* of accounting

When the strategizer hands work to a specialist, that is a **delegation**: an id (`D001`,
`D002`, ...), a task description, and a report that comes back.

The important half is the second one. Every real evaluation is attributed to the delegation that
produced it, so the system can answer, months later, *where did this number come from?*

## Concept 3: A metered oracle and an append-only ledger

The **evaluator** is ground truth: the function that scores a design. It is the same
`DataGenerator` you wrote in Lecture 19, the expensive function `f3dasm` calls to fill in an
unfinished job, with one addition: **every call is counted.**

The cap lives in the study's `config.yaml` as `eval_budget: 200`, a soft limit: when the run
approaches it the strategizer is nudged, never stopped.

It is the *only* metered path. Surrogates and optimizers the agents build on top are not metered:
fitting a GP a thousand times costs us nothing we care about.

## The ledger is an `ExperimentData`

Every real evaluation is written once, under a lock, to a shared canonical store, and that store
is an `ExperimentData`, the same object from Lecture 17.

Two things are added:

* **provenance columns** the framework stamps itself: `_delegation_id`, `_source`, `_ts`
* **protection**: a write that would shrink the store, or reset a completed evaluation, is refused

You have met the second idea already. In Lecture 19 you saw that marking a job `finished` means
`f3dasm` will *not* recompute that sample. The ledger takes that job-state discipline and makes
it one-directional: results go in, they do not come out.

## The real thing

The study in Lecture 19.2 left behind its evaluation stores. One of them ships with this
notebook: **538 real Abaqus evaluations** from its design-validation campaign.

Open it with the API you already know.

In [3]:
# This lecture's folder contains an experiment_data/ store, exactly like the ones you wrote
# in Lecture 17. from_file() takes the folder that CONTAINS experiment_data/, hence '.'
ledger = ExperimentData.from_file('.')

print("evaluations on the record:", len(ledger))

design, result = ledger.to_pandas()   # the same to_pandas() you used in Lectures 17-19
print("design variables:", list(design.columns)[:6], "...")

evaluations on the record: 538
design variables: ['circular', 'ratio_a', 'ratio_b', 'ratio_pitch', 'ratio_top_diameter', 'ratio_shear_modulus'] ...


In [4]:
# The provenance columns are stamped by the framework, not written by the agent.
provenance = [c for c in result.columns if c.startswith('_')]
print("provenance columns:", provenance)

print("\nevaluations per delegation:")
print(result['_delegation_id'].value_counts().to_string())

provenance columns: ['_delegation_id', '_source', '_ts', '_wall_ms']

evaluations per delegation:
_delegation_id
D000    538


Every row is stamped `D000`: this store is a single delegation's validation batch, not the whole
23-run campaign. That is worth stating rather than glossing: the point of provenance is that it
tells you what you actually have.

## In-class Exercise 1

The **reproduction gate** re-derives a run's headline number from the ledger and checks it against
the claim. Do that job by hand.

In the store you just loaded, find **the best feasible design**. A design is feasible only if all
of these hold. Notice where each rule comes from, because that distinction is the whole lecture:

1. `max_compressive_strain >= 0.80`  (Bessa 2019: physics)
2. `max_local_strain <= 0.02`  (Bessa 2019: physics)
3. `coilable == 1` and `riks_converged == True`  (the solver: a design that never converged has
   not been measured)
4. `ring_passthrough == 0`  (ours: added on 2026-07-20, three weeks in, after watching material
   pass through a ring. No physics required it. We decided it.)

Report its `sigma_crit` and compare it to the Bessa (2019) reference of **0.1306 kPa**.

Then answer: what would you have reported if you had simply taken the largest `sigma_crit` in the
store?

In [5]:
# Write your code for Exercise 1:



# until here.

<details>
<summary><b>Click here for Solution of In-class Exercise</b></summary>

```python
d = design.join(result)

feasible = d[(d.coilable == 1)
             & (d.riks_converged == True)
             & (d.max_compressive_strain >= 0.80)
             & (d.max_local_strain <= 0.02)
             & (d.ring_passthrough == 0)]

print("feasible designs:", len(feasible), "of", len(d))

best = feasible.loc[feasible.sigma_crit.idxmax()]
print("best feasible sigma_crit = %.4f kPa  (%.2f x Bessa)"
      % (best.sigma_crit, best.sigma_crit / 0.1306))

print("largest sigma_crit ignoring feasibility = %.4f kPa" % d.sigma_crit.max())
```

Expected output: 138 of 538 designs are feasible; the best feasible design reaches
**0.1106 kPa, which is 0.85x Bessa: it does not beat the baseline.** The largest
`sigma_crit` in the store is **144.7122 kPa**, about 1300 times larger, and it is not an
answer to anything: those designs are not coilable, or they never converged, or they violate
the strain limit.

The gap between those two numbers is the entire reason the rest of this lecture exists.
</details>

138 of 538 designs are feasible. The best of them reaches **0.1106 kPa, or 0.85× Bessa.**
It does not beat the baseline.

The largest `sigma_crit` in the store is **144.7 kPa**: about 1300× bigger, and not an answer to
anything.

Let the room sit with this before moving on. Everyone's first instinct is `.max()`, and the
answer that instinct produces is wrong by three orders of magnitude. Every mechanism in the rest
of the lecture (the charter, the critic, the gate) exists to stop a system (or a student) from
reporting 144.7.

## Concept 4: Falsification instead of a reward

With a reference solution you can say how wrong you are. Here you can rank two designs, but you
cannot say how far either sits from the best one, and an LLM asked to judge its own work is a
fluent, confident, unreliable referee.

So we did not give the system a reward to maximize. We gave it a **standard of proof**.

A *hypothesis* is a claim with a registered prediction and a verdict. Four statuses, no others:
`OPEN` → `SUPPORTED` / `FALSIFIED` / `INCONCLUSIVE`.

The rules live in one file, quoted verbatim into both the strategizer's and the critic's
instructions, so either can cite a clause and the other defers to the same words.

## The falsification charter

One file, quoted verbatim into both the strategizer's and the critic's instructions. Six numbered
clauses; two of them carry the argument.

> **§2** Before a hypothesis may be closed, an adequate falsification *attempt* must have been
> made: a **severe** test, one that genuinely probes the registered prediction and could have
> refuted the claim had it been false.

The charter in full, for reference:

```
§1  A hypothesis is ONE falsifiable claim carrying a registered prediction -
    the observable whose occurrence would refute the claim.

§2  ATTEMPT and VERDICT are distinct. Before a hypothesis may be closed, an
    adequate falsification ATTEMPT must have been made - a SEVERE test: one
    that genuinely probes the registered prediction and could have refuted the
    claim had it been false (a token probe is not adequate).
    [...] For a prediction whose refutation turns on FINDING an instance,
    severity means the search had the POWER to find that instance had it
    existed. A search that merely stopped improving is an INADEQUATE test:
    failing to find a better instance is not the same as showing none exists.

§3  A hypothesis is FALSIFIED if and only if an ADEQUATE test of its registered
    prediction yields a contradiction:
      - adequate test, prediction contradicted     -> FALSIFIED (you may not
        decline the verdict to protect a favoured claim);
      - adequate test, prediction NOT contradicted -> SUPPORTED;
      - inadequate or confounded test              -> INCONCLUSIVE. A
        contradiction from a flawed test indicts the test, not the hypothesis
        (Duhem-Quine).

§4  No moving the goalposts. A FALSIFIED verdict must rest on the contradiction
    of the SAME prediction that was registered - not a post-hoc observation
    chosen after seeing the data (the Texas-sharpshooter fallacy).

§5  SUPPORTED is corroboration, not proof. You never "confirm" a hypothesis;
    you only fail to falsify it.

§6  OPEN = no adequate test yet. The three closing statuses must cite a real
    delegation ID plus a CONCRETE result from it that bears on the registered
    prediction. What is forbidden is closing on prose or vibes.
```

## §2 is the clause you already know

> *A search that merely stopped improving is an INADEQUATE test: failing to find a better
> instance is not the same as showing none exists.*

<br>

This is the most common false conclusion in optimization: *I searched and found nothing better,
therefore nothing better exists.*

You know why that fails, from Lectures 13-16. A GP's predictive variance is *meant* to tell you
where you have not looked, and in Lecture 14 you watched it lie: fix `l = 10` with the optimizer
off and the GP is confidently flat across a region it never sampled.

So §2 asks the harder question. Not "was my model uncertain anywhere", but *what power did my
search actually have?*

§2 turns that intuition into an admissibility rule: to close an existence claim you must argue
your search *had the power to find* the thing, by coverage, by a surrogate that predicts the
claim's own observable above chance, or by a theoretical bound.

## §3: when a test fails, what is refuted?

> **§3** A contradiction from a flawed test indicts the test, not the hypothesis.

Philosophers call this the **Duhem-Quine problem**: you never test one assumption on its own, you
test the whole stack at once. When your GP disagrees with your simulation, one of them is wrong
and the data alone will not tell you which.

That is why `INCONCLUSIVE` is a verdict and not a shrug.

This is the pair's best link back to the probabilistic half of the course. Spend a moment on it:
"my BO run plateaued" is not evidence of a global optimum, and the students have the machinery to
say exactly why.

Also worth mentioning: writing this charter was not a software task. It went through several
internally inconsistent drafts, and one version routed the same outcome to INCONCLUSIVE in one
clause and SUPPORTED in another. Two agents then cited the same broken text at each other.

## Concept 5: A gate the run cannot talk its way through

The output of a run is a Jupyter notebook, `pipeline.ipynb`. Not a summary written afterwards:
its markdown cells are the write-up, and its code cells re-derive the headline result from the
ledger: the exercise you just did, run automatically.

Before a run may close, that notebook is executed end to end in a clean sandbox and its number is
compared against the claim. A run that cannot reproduce its own headline does not pass.

The outcome is recorded as `GATED` (passed the critic and reproduces) or `UNGATED`. If you check
one thing about a run, check that.

## Why a gate *and* a critic

An LLM critic can be argued with. A notebook that must run in a clean sandbox cannot.

* the claim overstates the evidence → caught by the **critic**
* the number cannot be regenerated → caught by the **gate**
* the narrative describes work that never ran → the **critic**, reading as a skeptical peer
* the notebook quietly hardcodes the answer → the **gate**, on a clean ledger

Neither alone is sufficient. The gate can tell you that a notebook runs and reproduces. It cannot tell you
whether the notebook is good science.

## Concept 6: Guards chosen by reversibility

Every time an agent does something you did not want, there is an obvious fix: block it. Do that a
dozen times and you have built a bureaucracy: a system that spends its budget being refused.

So we classify every guard by **reversibility**:

* **PROCEED + TIP**: easily reversible: let it through, say what looked off
* **CONFIRM (two-shot)**: reversible but weighty: refuse once with an explanation; an identical
  second call proceeds
* **PRECONDITION-BLOCK**: impossible until the world changes: refuse, and say what must change

In [6]:
def two_shot_confirm():
    """Refuse once with a reason, then honour the intent - carrying the caveat forward."""
    pending = set()

    def close_hypothesis(hyp_id, *, has_falsification_attempt):
        if has_falsification_attempt:
            return f"{hyp_id}: SUPPORTED (attempt on record)"
        if hyp_id not in pending:
            pending.add(hyp_id)
            return (f"{hyp_id}: refused once. Charter §5 - SUPPORTED means the claim "
                    f"survived an attempt to refute it, and none is on record. "
                    f"Re-call identically to confirm, with a written justification.")
        return f"{hyp_id}: SUPPORTED, recorded WITH the missing-attempt caveat attached"

    return close_hypothesis


close = two_shot_confirm()
print(close("H4", has_falsification_attempt=False))
print()
print(close("H4", has_falsification_attempt=False))   # identical second call

H4: refused once. Charter §5 - SUPPORTED means the claim survived an attempt to refute it, and none is on record. Re-call identically to confirm, with a written justification.

H4: SUPPORTED, recorded WITH the missing-attempt caveat attached


Note what the second call does *not* do: it does not pretend the attempt happened. It proceeds and
carries the caveat into the record. The guard makes a choice visible and attributable rather than
winning the argument.

The floor that stays hard, and why: schema validation, graph connectivity, notebook validity, the
per-delegation memory cap, reproduction-must-run, and closing-a-hypothesis-requires-evidence.
Provenance and host safety. Everything else was re-examined and most of it became a nudge.

The clearest conversion: a cap of three simultaneously-open hypotheses. That was closure
discipline, not safety, and the moment we asked the system to explore several design families at
once, three was simply wrong.

## Lesson 1: Do not overfit the prompt

When a bug is "fixed" by adding a rule to an agent's prompt, that rule must pass a test:

> Would a philosopher of science, reading this rule in isolation, nod at it as a general
> methodological principle, or frown at it as a workaround for one case?

A philosopher nods at: *"A hypothesis cannot be marked SUPPORTED without a falsification attempt
on record."* Popperian; applies to every run that will ever happen.

A philosopher frowns at: *"When calling HypothesisUpdate, pass a single ID, not a comma-separated
list."* That patches one tool-call error and names no principle.

The fallback matters as much as the test. When a failure is real but the obvious rule is overfit,
state the underlying principle instead, or fix it in **code** (a validation, an assertion at the
tool boundary), not in the prompt.

## Lesson 2: "Bulletproof" has a precise meaning

We once let each design space own its own physical data store. Then every place that counts
evaluations had to remember to aggregate across stores.

It didn't: in **seven** places, each found one validation run at a time. Then an eighth: the
design space could be chosen at the call site independently of the delegation, so the
registry-keyed fix was blind whenever the two diverged.

The root cause was not any of the eight sites. The data model did not match the question the whole
codebase asks: *how many evaluations in this run?*

Every aggregation helper was a band-aid: it made the right read available while leaving the wrong
read still present and still looking correct. So the next consumer was born blind.

> A mechanism is bulletproof only when the wrong usage is impossible or loud, not when the right
> usage is merely available.

This generalizes far beyond agents. It is a claim about API design, and you will meet it again the
first time you add an optional argument that callers must remember to pass.

Ask the room: how many of you have written a helper called `get_all_x()` alongside an existing
`get_x()`? That is the shape of the mistake. The fix is not a better helper; it is making
`get_x()` correct so there is nothing to remember.

## Lesson 3: Self-reports are leads, not diagnoses

Every node writes a retrospective when a run closes: what contradicted itself, the most uncertain
decision, what blocked it. These are the highest-signal artifact we have.

And they can be wrong about mechanism. One run blamed a *"silent Abaqus crash"* for a 63%
evaluation failure rate. The raw logs showed the **license server saturating at 16-way
concurrency**.

Same symptom, different fix, and a fix aimed at the reported cause would have achieved nothing
while looking like diligence. Verify the mechanism against the raw logs before acting on it.

In [7]:
# Lesson 3 on the real store: what a report SAYS versus what the record SHOWS.
d = design.join(result)
failed = ~d.riks_converged.astype(bool)

print("solves that never converged:", int(failed.sum()), "of", len(d))
print("failure rate:", round(100 * failed.mean(), 1), "%")

# A retrospective blamed 'a silent crash'. The timestamps say something else:
print("\nfailures by hour:")
print(d.loc[failed, '_ts'].str[:13].value_counts().head())

solves that never converged: 154 of 538
failure rate: 28.6 %

failures by hour:
_ts
2026-08-04T01    123
2026-08-04T13     27
2026-08-04T14      4
Name: count, dtype: int64


Same symptom, different mechanism. The retrospective is a lead. The `_ts` column is the
diagnosis.

## Lesson 4: Never let a cost argument masquerade as physics

Our problem statement described a solve-time cap as *"a hard property"*, and said a design family
that cannot fit inside it *"is not searchable"*. Both justifications for that cap were **cost**
arguments.

The consequence: a run closed with 5.7 of its 12 hours unspent, reasoning correctly, given what
it had been told, that the one remaining escape required a solver regime the infrastructure
cannot afford.

Presenting a budgeting default as a property of the oracle turned an accounting choice into a
boundary on the design space. The agent then reasoned honestly about a boundary that did not
exist.

The most instructive failure in the project, because nothing malfunctioned. The agent's own
retrospective drew exactly the right distinction: "'My search stopped improving' would not have
justified closing; 'the one remaining mechanism requires a solver regime the infrastructure
cannot afford' is a statement about the space and the tooling, not about my search."

That is better epistemics than most of us apply. It reached a wrong conclusion because *we* wrote
a false premise into the brief. The lesson is about us, not it.

## And one about the brief itself

> Write the problem statement the way a PI would brief a first-year graduate student.

A first-year student has freedom, and asks questions when something is unclear. What the brief
*must* pin down:

* **objective and success criteria**: the headline number or claim
* **design space**: every variable, with bounds, type, and units
* **what "valid" means**: feasibility limits, regimes of validity, thresholds

That middle bullet is a `Domain`. `add_float(name, low, high)`, `add_int`, `add_category`: you
wrote one in Lecture 17, and it is exactly what a good brief has to specify in prose.

What the brief should *not* do is prescribe the method. A brief that specifies the sampler and the
surrogate has hired a technician, not a researcher.

## Getting started: one folder

Everything adda needs is a directory. The only required file is the brief.

```
my_study/
  PROBLEM_STATEMENT.md   # required: the brief the agents work from
  config.yaml            # optional: model, budget, how a design is scored
  workspace/
    evaluator.py         # optional: your DataGenerator, if you ship one
```

```python
from a3dasm import AgenticRun
report = AgenticRun(study_dir="my_study").execute()
```

## The brief is the whole task definition

This is the real `PROBLEM_STATEMENT.md` that ships with the package, abridged. A deliberately
trivial problem, so the contract is visible:

```markdown
# Minimise a 2-D quadratic

## Objective
Minimise y = (x1 - 1)^2 + (x2 + 2)^2.
The global minimum is y = 0 at (x1, x2) = (1, -2).

## Success criteria
Report the argmin and the ledgered y* in pipeline.ipynb, derived from
ledgered rows, not a hardcoded number.

## Design space
| variable | type | bounds | units |
|---|---|---|---|
| x1 | continuous | [-5, 5] | dimensionless |
| x2 | continuous | [-5, 5] | dimensionless |
```

Objective, success criteria, design space with bounds and units. That table is a `Domain`,
written in prose.

## The other two files

`config.yaml`, six lines:

```yaml
model: claude-haiku-4-5-20251001
backend: claude
eval_budget: 200
evaluator:
  entrypoint: "workspace/evaluator.py:evaluate"
  output_names: [y]
```

`workspace/evaluator.py`, the ground truth:

```python
def evaluate(x1: float, x2: float) -> float:
    return (x1 - 1.0) ** 2 + (x2 + 2.0) ** 2
```

For the mast, that `evaluate` is an Abaqus job that takes minutes to hours. Everything else about
the study folder is identical.

## What comes back

* **`pipeline.ipynb`**, the deliverable, which reproduces its own headline number
* **`runs/<timestamp>/experiment_data/`**, every real evaluation
* **`runs/<timestamp>/run_status.json`**, `GATED` or not

That middle path is the same `experiment_data/` store you wrote in Lecture 17, and
`ExperimentData.from_file()` opens it, which is exactly what you did in today's exercise.

## Summary

* The architecture followed from three facts about the problem: **no reference solution**,
  **expensive and flaky evaluations**, and a **design space with no fixed boundary**.
* **Falsification replaced a reward**, and §2 of the charter is the Lecture 13-16 idea that a
  search must have *power* before its silence means anything.
* **Provenance is not optional**: the ledger is an `ExperimentData`, append-only and stamped.
* **The gate is mechanical**: it is the exercise you did today, run automatically.
* Guards by **reversibility**, so the framework nudges rather than substitutes its judgement for
  the scientist's.
* And: don't overfit the prompt, make the wrong usage loud, verify self-reports against raw logs,
  and never write a cost argument into a brief as though it were physics.

## Next lecture

We point it at a real research problem: the **supercompressible metamaterial** of Bessa,
Glowacki & Houlder (2019), which is also the problem of your **final project** (Lecture 20).
We go through what happened across 23 runs and 28 design ideas.

Including the headline it almost reported, the twelve times we changed the rules underneath it,
and the honest answer to *"did it beat the human?"*

### See you next class

Have fun!